<a href="https://colab.research.google.com/github/mikecrv2019-bit/MAESTRIA-IA/blob/main/MAI540_Modulo%204_Tarea%204.2_Herramienta%20de%20Regresi%C3%B3n%20Auditada/Herramienta_Regresion_CO2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Herramienta de Regresión Auditada — Emisiones de CO₂ de vehículos

**MAI 540 — Machine Learning | Módulo 4 | Tarea 4.2**

La herramienta estima las emisiones de CO₂ (g/km) de una configuración de vehículo ligero a partir de sus especificaciones técnicas. Apoya esta decisión: antes de la homologación oficial, un fabricante o importador decide si la configuración cumple el límite de emisiones de su flota. El error se expresa en **g/km**.

Las reglas de datos (variables prohibidas, partición, subgrupos) están en `context.md`, en esta misma carpeta.

## Decisiones del plan (paso 2)

| Decisión | Valor | Quién la tomó |
|---|---|---|
| Variables prohibidas por fuga | `Fuel Consumption Comb (L/100 km)`, `Comb (mpg)`, `City (L/100 km)`, `Hwy (L/100 km)` | Usuario (Comb y mpg en el paso 1; City y Hwy en el paso 2) |
| Variables excluidas | `Make`, `Model` | Usuario |
| Duplicados exactos | Se eliminan antes de la partición (7.385 → 6.282 filas) | Usuario |
| Partición | 75/25, `random_state=42`, estratificada por `Fuel Type` | Propuesta de Claude, aprobada por el usuario |
| Fila de gas natural (N, 1 fila) | Se conserva y va solo al entrenamiento; en prueba es "no evaluable" | Usuario |
| Imputación | Mediana (numéricas) y moda (categóricas). Hoy no hay nulos; los imputadores protegen ante datos nuevos | Claude |
| Codificación | `OneHotEncoder(handle_unknown="ignore")`; `Transmission` se codifica tal cual (27 valores) | Claude |
| Escalado | `StandardScaler` en las numéricas | Claude |
| Segundo modelo | `RandomForestRegressor(n_estimators=300, random_state=42)`; el resto por defecto, sin ajustar con el conjunto de prueba | Claude |

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from urllib.parse import quote

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

RANDOM_STATE = 42
TARGET = "CO2 Emissions(g/km)"
SUBGRUPO = "Fuel Type"
PROHIBIDAS = [
    "Fuel Consumption Comb (L/100 km)",   # fuga: el CO2 se calcula a partir de este consumo
    "Fuel Consumption Comb (mpg)",        # fuga: la misma medida en otra unidad
    "Fuel Consumption City (L/100 km)",   # fuga: misma prueba de consumo que el CO2
    "Fuel Consumption Hwy (L/100 km)",    # fuga: misma prueba de consumo que el CO2
]
EXCLUIDAS = ["Make", "Model"]
NUMERICAS = ["Engine Size(L)", "Cylinders"]
CATEGORICAS = ["Vehicle Class", "Transmission", "Fuel Type"]

## 1. Carga de datos (solo lectura)

Si el CSV está junto al cuaderno (repositorio clonado), se lee de `data/raw/`. En Google Colab se descarga desde el repositorio en GitHub.

In [2]:
RUTA_LOCAL = Path("data/raw/CO2 Emissions_Canada.csv")
URL_GITHUB = ("https://raw.githubusercontent.com/mikecrv2019-bit/MAESTRIA-IA/main/"
              + quote("MAI540_Modulo 4_Tarea 4.2_Herramienta de Regresión Auditada")
              + "/data/raw/" + quote("CO2 Emissions_Canada.csv"))

fuente = RUTA_LOCAL if RUTA_LOCAL.exists() else URL_GITHUB
df_raw = pd.read_csv(fuente)
print("Fuente:", fuente)
print("Filas y columnas:", df_raw.shape)
assert df_raw.shape == (7385, 12), "El CSV no es el esperado"
df_raw.head()

Fuente: data\raw\CO2 Emissions_Canada.csv
Filas y columnas: (7385, 12)


,Make,Model,Vehicle Class,Engine Size(L),Cylinders,Transmission,Fuel Type,Fuel Consumption City (L/100 km),Fuel Consumption Hwy (L/100 km),Fuel Consumption Comb (L/100 km),Fuel Consumption Comb (mpg),CO2 Emissions(g/km)
0,ACURA,ILX,COMPACT,2.0,4,AS5,Z,9.9,6.7,8.5,33,196
1,ACURA,ILX,COMPACT,2.4,4,M6,Z,11.2,7.7,9.6,29,221
2,ACURA,ILX HYBRID,COMPACT,1.5,4,AV7,Z,6.0,5.8,5.9,48,136
3,ACURA,MDX 4WD,SUV - SMALL,3.5,6,AS6,Z,12.7,9.1,11.1,25,255
4,ACURA,RDX AWD,SUV - SMALL,3.5,6,AS6,Z,12.1,8.7,10.6,27,244


## 2. Eliminación de duplicados exactos

Se eliminan las filas idénticas en las 12 columnas originales, **antes** de la partición. Así la misma fila no puede quedar a la vez en entrenamiento y en prueba. No es una transformación que estime parámetros: no genera fuga.

In [3]:
df = df_raw.drop_duplicates().reset_index(drop=True)
print("Duplicados exactos eliminados:", len(df_raw) - len(df))
print("Filas restantes:", len(df))
assert len(df) == 6282
df[SUBGRUPO].value_counts()

Duplicados exactos eliminados: 1103
Filas restantes: 6282


Fuel Type
X    3039
Z    2765
E     330
D     147
N       1
Name: count, dtype: int64

## 3. Eliminación de variables prohibidas y excluidas; separación de la objetivo

In [4]:
X = df.drop(columns=PROHIBIDAS + EXCLUIDAS + [TARGET])
y = df[TARGET]

assert not set(PROHIBIDAS + EXCLUIDAS) & set(X.columns), "Quedó una variable prohibida o excluida"
assert set(X.columns) == set(NUMERICAS + CATEGORICAS)
print("Predictoras:", list(X.columns))
print("Nulos en X:", int(X.isna().sum().sum()), "| Nulos en y:", int(y.isna().sum()))

Predictoras: ['Vehicle Class', 'Engine Size(L)', 'Cylinders', 'Transmission', 'Fuel Type']
Nulos en X: 0 | Nulos en y: 0


## 4. Partición entrenamiento / prueba (una sola, para los tres modelos)

Se estratifica por `Fuel Type` con las filas X, Z, E y D. La fila de gas natural (N) no se puede estratificar porque es una sola, así que se agrega solo al entrenamiento.

In [5]:
es_n = X[SUBGRUPO] == "N"
X_train, X_test, y_train, y_test = train_test_split(
    X[~es_n], y[~es_n], test_size=0.25, random_state=RANDOM_STATE, stratify=X.loc[~es_n, SUBGRUPO]
)
X_train = pd.concat([X_train, X[es_n]])
y_train = pd.concat([y_train, y[es_n]])

assert X_train.index.intersection(X_test.index).empty, "Hay filas compartidas entre entrenamiento y prueba"
assert len(X_train) + len(X_test) == len(X)
assert (X_test[SUBGRUPO] != "N").all()
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
pd.DataFrame({"entrenamiento": X_train[SUBGRUPO].value_counts(),
              "prueba": X_test[SUBGRUPO].value_counts()}).fillna(0).astype(int)

Entrenamiento: (4711, 5) | Prueba: (1571, 5)


,entrenamiento,prueba
Fuel Type,,
D,110,37
E,248,82
N,1,0
X,2279,760
Z,2073,692


## 5. Preprocesamiento dentro de un Pipeline

Imputación, codificación y escalado viven en un `ColumnTransformer` dentro de cada `Pipeline`. Por eso sus parámetros (medianas, modas, categorías, medias y desviaciones) se estiman solo cuando el pipeline se ajusta con el conjunto de entrenamiento.

In [6]:
def crear_preprocesador():
    numerico = Pipeline([
        ("imputar", SimpleImputer(strategy="median")),
        ("escalar", StandardScaler()),
    ])
    categorico = Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numerico, NUMERICAS),
        ("cat", categorico, CATEGORICAS),
    ])

## 6. Entrenamiento de los tres modelos

Los tres usan el mismo preprocesamiento y la misma partición:

- **(a) Referencia trivial:** `DummyRegressor` que siempre predice la media del entrenamiento.
- **(b) Regresión lineal.**
- **(c) Random Forest Regressor:** capta interacciones no lineales (por ejemplo, tipo de combustible × cilindrada) que la regresión lineal no representa, y no depende del escalado.

In [7]:
modelos = {
    "Referencia (media)": DummyRegressor(strategy="mean"),
    "Regresión lineal": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
}

pipelines, pred_train, pred_test = {}, {}, {}
for nombre, modelo in modelos.items():
    pipe = Pipeline([("prep", crear_preprocesador()), ("modelo", modelo)])
    pipe.fit(X_train, y_train)          # ajuste SOLO con entrenamiento
    pipelines[nombre] = pipe
    pred_train[nombre] = pipe.predict(X_train)
    pred_test[nombre] = pipe.predict(X_test)  # prueba: solo transformar y predecir
    print(f"{nombre}: entrenado")

Referencia (media): entrenado


Regresión lineal: entrenado


Random Forest: entrenado


### Verificaciones del ajuste sin fuga

In [8]:
# El escalador aprendió la media del ENTRENAMIENTO, no la del dataset completo
escalador = pipelines["Regresión lineal"].named_steps["prep"].named_transformers_["num"].named_steps["escalar"]
assert np.allclose(escalador.mean_, X_train[NUMERICAS].mean().values)
print("Media aprendida por el escalador:", escalador.mean_.round(4))
print("Media del entrenamiento:         ", X_train[NUMERICAS].mean().values.round(4))
print("Media del dataset completo:      ", X[NUMERICAS].mean().values.round(4))

# El modelo de referencia predice siempre la media del entrenamiento
assert np.allclose(pred_test["Referencia (media)"], y_train.mean())
print("Predicción constante de la referencia:", round(y_train.mean(), 2), "g/km")

for nombre in modelos:
    assert len(pred_test[nombre]) == len(y_test)
print("Predicciones de prueba por modelo:", {k: len(v) for k, v in pred_test.items()})

Media aprendida por el escalador: [3.1562 5.6101]
Media del entrenamiento:          [3.1562 5.6101]
Media del dataset completo:       [3.1618 5.6189]
Predicción constante de la referencia: 250.79 g/km
Predicciones de prueba por modelo: {'Referencia (media)': 1571, 'Regresión lineal': 1571, 'Random Forest': 1571}


---
Las métricas (MSE, RMSE y R²), el análisis de residuos, la auditoría y las limitaciones corresponden a los pasos 3 a 7 y se agregan en las secciones siguientes.